# Generador de clientes sintéticos pensionarios de México

Crea una base de personas ficticias (IMSS Ley 73 / Ley 97, ISSSTE Cuentas Individuales e independientes sin afiliación) a partir de los CSV del directorio de entrada. No utiliza ni produce datos personales reales.

- **Parámetros:** todo lo ajustable (rutas, semilla, `N_CLIN`, fecha de corte, proporción por perfil, distribuciones de edad, salario, saldos, probabilidades y montos legales de referencia) vive en `config/parametros.py`.
- **Lógica:** paquete `generador/`. Cada cliente se genera por perfil, se clasifica con reglas pensionarias y se valida al 100 %.
- `N_CLIN` es exactamente el número de clientes a generar.
- Los importes están en pesos constantes de la fecha de corte. Todas las resoluciones son simuladas.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona si Jupyter se abre desde la raíz del proyecto o desde esta carpeta.
RUTA_NOTEBOOK = next(
    (ruta for ruta in (Path.cwd(), Path.cwd() / 'SyntethicDataGenerator') if (ruta / 'config').is_dir()),
    None,
)
if RUTA_NOTEBOOK is None:
    raise FileNotFoundError('No se encontró el directorio config junto al notebook.')
sys.path.insert(0, str(RUTA_NOTEBOOK))

# Descarta módulos en caché para que los cambios en config/ se apliquen al volver a ejecutar.
for modulo in [m for m in sys.modules if m.split('.')[0] in ('config', 'generador')]:
    del sys.modules[modulo]

import config as cfg
from generador import generar_clientes

cfg.DIR_OUTPUT.mkdir(exist_ok=True)
rng = np.random.default_rng(cfg.SEMILLA)

In [2]:
nombres = pd.read_csv(cfg.ARCHIVO_NOMBRES, encoding=cfg.CODIFICACION_ENTRADA)
apellidos = pd.read_csv(cfg.ARCHIVO_APELLIDOS, encoding=cfg.CODIFICACION_ENTRADA)

# Controles de calidad de los insumos.
assert len(nombres) == cfg.NOMBRES_ESPERADOS, f'Se esperan {cfg.NOMBRES_ESPERADOS} nombres.'
assert set(nombres['genero']) == set(cfg.GENEROS_ARCHIVO), 'Géneros inesperados en el archivo de nombres.'
assert len(apellidos) == cfg.APELLIDOS_ESPERADOS, f'Se esperan {cfg.APELLIDOS_ESPERADOS} apellidos.'

display(nombres.groupby('genero').size().rename('cantidad'))
display(apellidos.head())

genero
Hombre    250
Mujer     250
Name: cantidad, dtype: int64

,id,apellido
0,1,Hernández
1,2,García
2,3,Martínez
3,4,López
4,5,González


In [3]:
# Genera historial laboral, saldos (salario, RCV, AV, vivienda), pensión estimada e identificadores.
clientes = generar_clientes(rng, nombres, apellidos)
clientes.head(10)

,id_cliente,nombre,ap_paterno,ap_materno,age,f_nacim,genero,entidad,nss,curp,...,anio_fil,sem_cot,salario_mensual,saldo_rcv,saldo_av,saldo_vivienda,av_mensual,regimen_estimado,ley_bandera,escenario_estimado
0,1,Claudio,Treviño,Escamilla,64,1962-06-30,H,AS,23886232752,TEEC620630HASRSL03,...,1988,1765,21229.48,848971.77,0.00,0.00,0.0,IMSS73,LEY_73,IMSS73_DERECHO_PENSION
1,2,Emmanuel,Domínguez,Meza,68,1958-06-18,H,ZS,None,DOME580618HZSMZM05,...,1995,908,20467.50,217296.49,0.00,211546.39,0.0,ISSSTE,LEY_ISSSTE,ISSSTE_NEGATIVA_PENSION
2,3,Esperanza,Ortega,Del Río,60,1966-09-20,M,QT,None,OERE660920MQTRXS05,...,1991,1365,10827.93,206829.14,0.00,0.00,0.0,ISSSTE,LEY_ISSSTE,ISSSTE_PENSION_GARANTIZADA
3,4,Emely,Labastida,Alvarado,62,1964-02-12,M,BC,55836442438,LAAE640212MBCBLM06,...,1983,1490,9730.99,152100.51,39662.32,200769.39,0.0,IMSS73,LEY_73,IMSS73_GARANTIZADA
4,5,Joel,Escobedo,Barajas,62,1963-12-12,H,QR,88006370618,EOBJ631212HQRSRL04,...,2000,1024,58776.12,1072632.64,0.00,714012.84,0.0,IMSS97,LEY_97,IMSS97_RECURSOS_PROPIOS
5,6,Adán,Ugalde,Palacios,60,1966-02-05,H,TS,02866639798,UAPA660205HTSGLD04,...,1986,1441,10078.38,205305.02,0.00,141599.60,0.0,IMSS73,LEY_73,IMSS73_GARANTIZADA
6,7,Edgar,Ortiz,Barrientos,65,1961-02-26,H,QT,56046146306,OIBE610226HQTRRD09,...,2004,882,18212.35,281787.21,28592.91,1577.14,0.0,IMSS97,LEY_97,IMSS97_PENSION_GARANTIZADA
7,8,Imelda,Godoy,Escobedo,60,1966-05-11,M,AS,27086674986,GOEI660511MASDSM00,...,2008,597,20802.88,147583.63,0.00,126419.45,0.0,IMSS97,LEY_97,IMSS97_NEGATIVA_PENSION
8,9,Leonel,Valdez,Marín,61,1965-01-09,H,DG,None,VAML650109HDGLRN07,...,1997,1433,35494.46,736799.86,0.00,252825.02,0.0,ISSSTE,LEY_ISSSTE,ISSSTE_RECURSOS_PROPIOS
9,10,Josué,Gutiérrez,Ponce,69,1957-01-22,H,SP,90975717266,GUPJ570122HSPTNS00,...,1997,1265,14438.55,467121.32,0.00,198077.86,0.0,IMSS97,LEY_97,IMSS97_PENSION_GARANTIZADA


In [5]:
# Control de calidad del layout final y de los identificadores sintéticos.
assert list(clientes.columns) == cfg.COLUMNAS_SALIDA
assert clientes['id_cliente'].is_unique
assert clientes['curp'].is_unique
assert clientes['correo'].is_unique
assert len(clientes) == cfg.N_CLIN
assert clientes['correo'].str.endswith(('@yahoo.com', '@gmail.com', '@hotmail.com', '@outlook.com')).all()
assert clientes.loc[clientes['nss'].notna(), 'nss'].str.fullmatch(r'\d{11}').all()
assert clientes['anios_cotizados'].ge(0).all()

print('Régimen y escenario estimado')
display(clientes.groupby(['regimen_estimado', 'escenario_estimado']).size().rename('clientes'))
display(pd.crosstab(pd.cut(clientes['age'], [0, 49, 59, 64, 69, 120], labels=['<50', '50-59', '60-64', '65-69', '70+']),
                    clientes['regimen_estimado']))
display(clientes[['age', 'anios_cotizados', 'sem_cot', 'salario_mensual', 'saldo_rcv', 'saldo_av', 'saldo_vivienda', 'av_mensual']].describe(percentiles=[.5, .9, .99]).round(1))
print(f"Clientes con AV = 0: {(clientes['saldo_av'] == 0).mean():.1%} | con vivienda = 0: {(clientes['saldo_vivienda'] == 0).mean():.1%}")

Régimen y escenario estimado


regimen_estimado  escenario_estimado        
IMSS73            IMSS73_DERECHO_PENSION        15000
                  IMSS73_GARANTIZADA            10000
                  IMSS73_NEGATIVA_PENSION        6000
IMSS97            IMSS97_NEGATIVA_PENSION        8000
                  IMSS97_PENSION_GARANTIZADA    17000
                  IMSS97_RECURSOS_PROPIOS        8000
ISSSTE            ISSSTE_NEGATIVA_PENSION        6000
                  ISSSTE_PENSION_GARANTIZADA    10000
                  ISSSTE_RECURSOS_PROPIOS       10000
NO_APLICA         INDEPENDIENTES                10000
Name: clientes, dtype: int64

regimen_estimado,IMSS73,IMSS97,ISSSTE,NO_APLICA
age,,,,
<50,0,0,0,4290
50-59,0,0,0,3894
60-64,23745,24939,16667,1194
65-69,7053,7970,8555,622
70+,202,91,778,0


,age,anios_cotizados,sem_cot,salario_mensual,saldo_rcv,saldo_av,saldo_vivienda,av_mensual
count,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
mean,62.0,19.7,1053.1,20892.0,413525.9,18342.8,132088.6,25.5
std,5.1,10.1,531.9,11857.8,377417.0,111310.5,168805.3,137.3
min,30.0,0.0,0.0,9582.0,0.0,0.0,0.0,0.0
50%,63.0,22.0,1163.0,17758.6,324152.8,0.0,70916.9,0.0
90%,66.0,30.0,1613.0,34921.8,919640.8,17808.0,370845.8,0.0
99%,70.0,36.0,1921.0,63332.8,1733347.7,457595.2,704759.0,765.4
max,74.0,45.0,2366.0,83554.7,4122611.8,5833928.0,1224236.0,1975.2


Clientes con AV = 0: 88.1% | con vivienda = 0: 35.4%


In [6]:
marca_tiempo = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S_%f')
carpeta_salida = cfg.DIR_OUTPUT / marca_tiempo
carpeta_salida.mkdir(parents=True, exist_ok=False)

archivos_por_regimen = {
    'IMSS97': 'clientes_IMSS97.csv',
    'IMSS73': 'clientes_IMSS73.csv',
    'ISSSTE': 'clientes_ISSSTE_Cuentas_Individuales.csv',
}
for regimen, nombre_archivo in archivos_por_regimen.items():
    grupo = clientes.loc[clientes['regimen_estimado'].eq(regimen)]
    ruta_archivo = carpeta_salida / nombre_archivo
    grupo.to_csv(ruta_archivo, index=False, encoding=cfg.CODIFICACION_SALIDA, date_format=cfg.FORMATO_FECHA)
    print(f'{len(grupo):,} registros: {ruta_archivo.resolve()}')

independientes = clientes['regimen_estimado'].eq(cfg.NO_APLICA).sum()
print(f'{independientes:,} independientes sin régimen contributivo no incluidos en los tres archivos.')

33,000 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261021_153927_012422/clientes_IMSS97.csv
31,000 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261021_153927_012422/clientes_IMSS73.csv
26,000 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261021_153927_012422/clientes_ISSSTE_Cuentas_Individuales.csv
10,000 independientes sin régimen contributivo no incluidos en los tres archivos.
